# RDKit shape and color scoring

This notebook compares the cosine and Tanimoto similarities of 12 molecular representations against maximum conformational overlap. \
Maximum conformational overlap is measured by RDKit's shape and color scores. The pipeline is as follows:

- Load cosine and Tanimoto similarity matrices 
- Find the minimum and maximum similarity observed over the 30K set of ZINC molecules
- Divide the observed range into 10 equal-width similarity intervals
- Uniformly sample up to 100 molecule pairs from every interval
- Generates conformer for the molecules used by that model
- Calculate the maximum shape-only RDShapeAlign score over all conformer pairs;
- Calculate the maximum color-only RDShapeAlign score over all conformer pairs;
- Calculate separate Spearman correlations between the sampled 2D similarities and the shape and color scores
- Create summary tables and a plotly figure

Shape and color are optimized independently:

- **Shape:** `useColors=False`, `opt_param=1.0`
- **color:** `useColors=True`, `opt_param=0.0`

The only cell you should need to edit is the file path section below.

## Imports

In [ ]:
from __future__ import annotations

import gc
import os
import re
import zlib
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import string
from IPython.display import display

from rdkit import Chem
from rdkit.Chem import AllChem, rdShapeAlign

## File paths

In [ ]:
# Path to the ZINC30K dataset
SMILES_PATH = "/path/to/ZINC 30K.csv"

# Folder paths for the ZINC30K pairwise similarity matrices. You will need to create the pairwise similarity matrices.
EMBEDDING_DIR   = Path("/path/to/ZINC30K/Cosine matrices/")
FINGERPRINT_DIR = Path("/path/to/ZINC30KTanimoto matrices/")

# Output directory paths
OUTPUT_DIR          = Path("./shape_color_similarity_equal_width_results")
PAIR_OUTPUT_DIR     = OUTPUT_DIR / "sampled_pairs"
INTERVAL_OUTPUT_DIR = OUTPUT_DIR / "interval_diagnostics"
FIGURE_OUTPUT_DIR   = OUTPUT_DIR / "figures"

## Set up

In [6]:
data       = pd.read_csv(SMILES_PATH)[:30000]
smiles     = data['SMILES'].tolist()

In [7]:
for directory in (
    OUTPUT_DIR,
    PAIR_OUTPUT_DIR,
    INTERVAL_OUTPUT_DIR,
    FIGURE_OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


#### Sampling ####
N_INTERVALS = 10
# Maximum requested sample size for each interval. Sparse intervals contribute all available pairs.
N_PAIRS_PER_INTERVAL = 100
# Number of rows/columns loaded per matrix block.
MATRIX_BLOCK_SIZE = 1024
# Fix a seed
BASE_SEED = 42

#### Conformer generation and sampling ####
N_CONFS               = 10
N_JOBS                = min(16, os.cpu_count() or 1)
PRUNE_RMS             = 0.5
PAIR_SCORE_CHUNK_SIZE = 25
ALIGNMENT_SCORE_DTYPE = np.float32

# color-only optimization. RDShapeAlign uses:
#   0.0 = color only
#   0.5 = equal shape and color
#   1.0 = shape only
COLOR_OPT_PARAM = 0.0


## Conformer generation and maximum shape/color RDShapeAlign scoring

Molecules that fail parsing, conformer generation, or optimization are counted. Pairs involving those molecules receive `NaN` for both shape and color.

For every valid conformer pair, two independent alignments are performed:

- a shape-only alignment, used to update the maximum shape score;
- a color-only alignment, used to update the maximum color score.

The pair-level output contains independent maxima for the two scoring objectives.

In [8]:
# Globals inside each pair-scoring worker process.
_WORKER_MOLS = None
_WORKER_CONFS = None


def make_conformers(
    smiles_string,
    n_confs=10,
    seed=1,
    prune_rms=0.5,
    rdkit_threads=1,
):
    """ Generate and optimize an RDKit conformer ensemble. """
    mol = Chem.MolFromSmiles(smiles_string)

    if mol is None:
        raise ValueError(f"Bad SMILES: {smiles_string}")

    mol = Chem.AddHs(mol)

    params                = AllChem.ETKDGv3()
    params.randomSeed     = int(seed)
    params.pruneRmsThresh = float(prune_rms)
    params.numThreads     = int(rdkit_threads)

    conf_ids = list(
        AllChem.EmbedMultipleConfs(
            mol,
            numConfs=int(n_confs),
            params=params,
        )
    )

    if not conf_ids:
        raise ValueError(
            f"No conformers generated for: {smiles_string}"
        )

    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMoleculeConfs(
            mol,
            numThreads=int(rdkit_threads),
        )
    else:
        AllChem.UFFOptimizeMoleculeConfs(
            mol,
            numThreads=int(rdkit_threads),
        )

    mol = Chem.RemoveHs(mol)

    conf_ids = [
        conformer.GetId()
        for conformer in mol.GetConformers()
    ]

    if not conf_ids:
        raise ValueError(
            "Conformers were lost after hydrogen removal for: "
            f"{smiles_string}"
        )

    return mol, conf_ids


def max_rdshapealign_shape_color_scores(
    mol_a,
    confs_a,
    mol_b,
    confs_b,
    color_opt_param=0.0,
):
    """
    Return independent maximum shape and color scores.

    Shape is optimized using:
        useColors=False, opt_param=1.0

    Color is optimized using:
        useColors=True, opt_param=color_opt_param

    With color_opt_param=0.0, the second alignment is color-only.
    """
    best_shape_score  = -1.0
    best_color_score  = -1.0

    for conf_a in confs_a:
        for conf_b in confs_b:
            # AlignMol modifies the probe molecule, so use a fresh copy
            # for each independent optimization.
            shape_probe = Chem.Mol(mol_b)

            shape_score, _ = rdShapeAlign.AlignMol(
                mol_a,
                shape_probe,
                int(conf_a),
                int(conf_b),
                False,  # useColors=False
                1.0,    # shape-only optimization
            )

            if shape_score > best_shape_score:
                best_shape_score = shape_score

            color_probe = Chem.Mol(mol_b)

            _, color_score = rdShapeAlign.AlignMol(
                mol_a,
                color_probe,
                int(conf_a),
                int(conf_b),
                True,                    # useColors=True
                float(color_opt_param),  # 0.0 = color-only
            )

            if color_score > best_color_score:
                best_color_score = color_score

    return (
        float(best_shape_score),
        float(best_color_score),
    )


def _prepare_one(args):
    molecule_index, smiles_string, n_confs, prune_rms = args

    mol, conf_ids = make_conformers(
        smiles_string,
        n_confs=n_confs,
        seed=int(molecule_index) + 1,
        prune_rms=prune_rms,
        rdkit_threads=1,
    )

    return int(molecule_index), mol, conf_ids


def _init_score_worker(mols, confs):
    global _WORKER_MOLS, _WORKER_CONFS
    _WORKER_MOLS = mols
    _WORKER_CONFS = confs


def _indexed_pair_chunks(
    pairs,
    valid_pair_indices,
    chunk_size,
):
    """ Yield chunks of (result index, molecule i, molecule j)."""
    for start in range(
        0,
        len(valid_pair_indices),
        int(chunk_size),
    ):
        end = min(
            start + int(chunk_size),
            len(valid_pair_indices),
        )

        chunk = []

        for pair_index in valid_pair_indices[start:end]:
            molecule_i, molecule_j = pairs[pair_index]

            chunk.append(
                (
                    int(pair_index),
                    int(molecule_i),
                    int(molecule_j),
                )
            )

        yield chunk


def _score_indexed_pair_chunk(
    pair_chunk,
    color_opt_param,
):
    results = []

    for pair_index, molecule_i, molecule_j in pair_chunk:
        shape_score, color_score = (
            max_rdshapealign_shape_color_scores(
                _WORKER_MOLS[molecule_i],
                _WORKER_CONFS[molecule_i],
                _WORKER_MOLS[molecule_j],
                _WORKER_CONFS[molecule_j],
                color_opt_param=color_opt_param,
            )
        )

        results.append(
            (
                pair_index,
                shape_score,
                color_score,
            )
        )

    return results


def max_rdshapealign_scores_for_pairs_parallel(
    smiles,
    pairs,
    n_confs=5,
    n_jobs=None,
    prune_rms=0.5,
    chunk_size=25,
    dtype=np.float32,
    color_opt_param=0.0,
    progress_prefix="",
):
    """
    Calculate maximum shape and color scores for supplied molecule pairs.

    Molecules that fail conformer generation are skipped. Both scores for
    pairs involving a failed molecule remain NaN.
    """
    if n_jobs is None:
        n_jobs = os.cpu_count() or 1

    pairs        = np.asarray(pairs, dtype=np.int64)
    smiles_array = np.asarray(smiles, dtype=object)

    used_molecule_indices = np.unique(pairs.ravel())

    # Dictionaries avoid passing a 30K-element mostly-empty list to every
    # alignment-scoring worker.
    mols  = {}
    confs = {}

    prep_tasks = [
        (
            int(index),
            smiles_array[int(index)],
            int(n_confs),
            float(prune_rms),
        )
        for index in used_molecule_indices
    ]

    failed_molecule_indices = set()

    conformer_description = (
        f"{progress_prefix} conformers".strip()
    )

    # Prepare all molecules used by this model's sampled pairs.
    with ProcessPoolExecutor(
        max_workers=int(n_jobs)
    ) as executor:
        future_to_task = {
            executor.submit(_prepare_one, task): task
            for task in prep_tasks
        }

        for future in tqdm(
            as_completed(future_to_task),
            total=len(future_to_task),
            desc=conformer_description,
            unit="molecule",
            leave=True,
        ):
            task = future_to_task[future]
            molecule_index, smiles_string, _, _ = task

            try:
                result_index, mol, conf_ids = future.result()

            except Exception as exc:
                failed_molecule_indices.add(
                    int(molecule_index)
                )

                tqdm.write(
                    f"Skipping molecule {molecule_index}: "
                    f"{smiles_string}\n"
                    f"  {type(exc).__name__}: {exc}"
                )
                continue

            mols[result_index] = mol
            confs[result_index] = conf_ids

    shape_scores = np.full(
        len(pairs),
        np.nan,
        dtype=dtype,
    )

    color_scores = np.full(
        len(pairs),
        np.nan,
        dtype=dtype,
    )

    valid_pair_indices = np.asarray(
        [
            pair_index
            for pair_index, (molecule_i, molecule_j)
            in enumerate(pairs)
            if int(molecule_i) in mols
            and int(molecule_j) in mols
        ],
        dtype=np.int64,
    )

    n_skipped_pairs = (
        len(pairs) - len(valid_pair_indices)
    )

    if failed_molecule_indices:
        tqdm.write(
            f"{progress_prefix}: skipped "
            f"{len(failed_molecule_indices)} molecule(s) and "
            f"{n_skipped_pairs} affected pair(s)."
        )

    if len(valid_pair_indices) == 0:
        tqdm.write(
            f"{progress_prefix}: no valid pairs remain to score."
        )

        return (
            pairs,
            shape_scores,
            color_scores,
        )

    pair_chunks = list(
        _indexed_pair_chunks(
            pairs=pairs,
            valid_pair_indices=valid_pair_indices,
            chunk_size=chunk_size,
        )
    )

    score_description = (
        f"{progress_prefix} shape + color scores".strip()
    )

    # Score pairs whose two molecules were prepared successfully.
    with ProcessPoolExecutor(
        max_workers=int(n_jobs),
        initializer=_init_score_worker,
        initargs=(mols, confs),
    ) as executor:
        futures = {
            executor.submit(
                _score_indexed_pair_chunk,
                chunk,
                float(color_opt_param),
            ): chunk
            for chunk in pair_chunks
        }

        with tqdm(
            total=len(valid_pair_indices),
            desc=score_description,
            unit="pair",
            leave=True,
        ) as progress:
            for future in as_completed(futures):
                chunk = futures[future]

                # Pair-alignment errors intentionally propagate.
                chunk_results = future.result()

                for (
                    pair_index,
                    shape_score,
                    color_score,
                ) in chunk_results:
                    shape_scores[pair_index] = shape_score
                    color_scores[pair_index] = color_score

                progress.update(len(chunk))

    return (
        pairs,
        shape_scores,
        color_scores,
    )


## Memory-bounded upper-triangle block utilities

Only the unique unordered molecule pairs `(i, j)` with `i < j` are processed.

In [ ]:
def count_upper_triangle_blocks(
    n_molecules,
    block_size,
):
    """ Return the number of square blocks on or above the diagonal."""
    n_blocks = (
        int(n_molecules) + int(block_size) - 1
    ) // int(block_size)

    return n_blocks * (n_blocks + 1) // 2


def iter_upper_triangle_blocks(
    similarity_matrix,
    block_size=1024,
):
    """
    Iterate over the unique upper triangle in matrix blocks.

    Yields a dictionary containing flattened values and enough indexing
    information to convert selected positions back to global (i, j) pairs.
    """
    n_molecules = int(similarity_matrix.shape[0])
    block_size = int(block_size)

    for row_start in range(
        0,
        n_molecules,
        block_size,
    ):
        row_end = min(
            row_start + block_size,
            n_molecules,
        )

        for col_start in range(
            row_start,
            n_molecules,
            block_size,
        ):
            col_end = min(
                col_start + block_size,
                n_molecules,
            )

            block = np.asarray(
                similarity_matrix[
                    row_start:row_end,
                    col_start:col_end,
                ]
            )

            n_block_cols = col_end - col_start

            if row_start == col_start:
                local_rows, local_cols = np.triu_indices(
                    row_end - row_start,
                    k=1,
                )

                values = np.asarray(
                    block[local_rows, local_cols]
                )

            else:
                local_rows = None
                local_cols = None
                values = np.asarray(block).reshape(-1)

            yield {
                "row_start": int(row_start),
                "col_start": int(col_start),
                "n_block_cols": int(n_block_cols),
                "values": values,
                "local_rows": local_rows,
                "local_cols": local_cols,
            }


def block_positions_to_pairs(
    positions,
    row_start,
    col_start,
    n_block_cols,
    local_rows,
    local_cols,
):
    """ Map flattened upper-triangle block positions to global pairs."""
    positions = np.asarray(
        positions,
        dtype=np.int64,
    )

    if local_rows is None:
        molecule_i = (
            int(row_start)
            + positions // int(n_block_cols)
        )

        molecule_j = (
            int(col_start)
            + positions % int(n_block_cols)
        )

    else:
        molecule_i = (
            int(row_start)
            + local_rows[positions]
        )

        molecule_j = (
            int(col_start)
            + local_cols[positions]
        )

    return np.column_stack(
        [molecule_i, molecule_j]
    ).astype(np.int64, copy=False)


def keep_smallest_priority_candidates(
    existing_pairs,
    existing_similarities,
    existing_priorities,
    candidate_pairs,
    candidate_similarities,
    candidate_priorities,
    sample_size,
):
    """
    Merge old and new candidates and retain the smallest random priorities.

    Giving every pair an independent continuous random priority and keeping
    the smallest k priorities is uniform sampling without replacement.
    """
    if existing_pairs.size == 0:
        merged_pairs = candidate_pairs
        merged_similarities = candidate_similarities
        merged_priorities = candidate_priorities

    else:
        merged_pairs = np.vstack(
            [existing_pairs, candidate_pairs]
        )

        merged_similarities = np.concatenate(
            [
                existing_similarities,
                candidate_similarities,
            ]
        )

        merged_priorities = np.concatenate(
            [
                existing_priorities,
                candidate_priorities,
            ]
        )

    keep_count = min(
        int(sample_size),
        len(merged_priorities),
    )

    if keep_count < len(merged_priorities):
        keep_indices = np.argpartition(
            merged_priorities,
            keep_count - 1,
        )[:keep_count]

    else:
        keep_indices = np.arange(
            len(merged_priorities)
        )

    return (
        merged_pairs[keep_indices],
        merged_similarities[keep_indices],
        merged_priorities[keep_indices],
    )


## Equal-width interval sampling

- The first full scan finds the true finite minimum and maximum of the matrix's unique upper triangle. Eleven equally spaced edges are then constructed, defining ten equal-width intervals. 
- The second scan counts every interval and retains a uniform sample of 100 pairs from each one.

In [ ]:
def find_upper_triangle_min_max(
    similarity_matrix,
    block_size=1024,
    description="matrix",
):
    """
    Find the finite minimum and maximum similarity in the complete unique
    upper triangle without materializing it.
    """
    n_molecules = int(similarity_matrix.shape[0])

    total_blocks = count_upper_triangle_blocks(
        n_molecules=n_molecules,
        block_size=block_size,
    )

    minimum_similarity = np.inf
    maximum_similarity = -np.inf

    n_finite = 0
    n_nonfinite = 0

    block_iterator = iter_upper_triangle_blocks(
        similarity_matrix=similarity_matrix,
        block_size=block_size,
    )

    for block in tqdm(
        block_iterator,
        total=total_blocks,
        desc=f"{description}: finding min/max",
        unit="block",
        leave=True,
    ):
        values = block["values"]

        finite_mask = np.isfinite(values)
        finite_values = values[finite_mask]

        n_finite += int(finite_mask.sum())
        n_nonfinite += int(
            values.size - finite_mask.sum()
        )

        if finite_values.size == 0:
            continue

        minimum_similarity = min(
            minimum_similarity,
            float(np.min(finite_values)),
        )

        maximum_similarity = max(
            maximum_similarity,
            float(np.max(finite_values)),
        )

    if n_finite == 0:
        raise ValueError(
            f"{description}: no finite upper-triangle "
            "similarities were found."
        )

    scan_info = {
        "minimum_similarity": float(
            minimum_similarity
        ),
        "maximum_similarity": float(
            maximum_similarity
        ),
        "finite_upper_triangle_pairs": int(
            n_finite
        ),
        "nonfinite_upper_triangle_pairs": int(
            n_nonfinite
        ),
    }

    return (
        minimum_similarity,
        maximum_similarity,
        scan_info,
    )


def assign_equal_width_intervals(
    values,
    interval_edges,
):
    """
    Assign values to interval IDs 0 through 9.

    The intervals are left-closed and right-open, except that the tenth
    interval includes the global maximum.
    """
    values = np.asarray(values)

    interval_edges = np.asarray(
        interval_edges,
        dtype=np.float64,
    )

    # Searching the nine internal edges directly returns 0 through 9.
    # side='right' places a value exactly equal to an internal edge in the
    # interval beginning at that edge.
    interval_ids = np.searchsorted(
        interval_edges[1:-1],
        values,
        side="right",
    )

    return interval_ids.astype(
        np.uint8,
        copy=False,
    )


def sample_pairs_from_equal_width_intervals(
    similarity_matrix,
    n_pairs_per_interval=100,
    n_intervals=10,
    block_size=1024,
    seed=42,
    description="matrix",
):
    """
    Divide the observed finite similarity range into equal-width intervals
    and sample up to the requested number of molecule pairs uniformly from every interval. 
    Sparse intervals contribute all available pairs.
    """

    (
        minimum_similarity,
        maximum_similarity,
        scan_info,
    ) = find_upper_triangle_min_max(
        similarity_matrix=similarity_matrix,
        block_size=block_size,
        description=description,
    )

    interval_edges = np.linspace(
        minimum_similarity,
        maximum_similarity,
        int(n_intervals) + 1,
        dtype=np.float64,
    )

    interval_width = (
        maximum_similarity - minimum_similarity
    ) / int(n_intervals)

    print(
        f"\n{description}: finite upper-triangle range = "
        f"{minimum_similarity:.8f} to "
        f"{maximum_similarity:.8f}"
    )

    print(
        f"{description}: equal interval width = "
        f"{interval_width:.8f}"
    )

    interval_labels = []

    for interval_id in range(int(n_intervals)):
        lower = interval_edges[interval_id]
        upper = interval_edges[interval_id + 1]

        closing_bracket = (
            "]"
            if interval_id == int(n_intervals) - 1
            else ")"
        )

        interval_labels.append(
            f"[{lower:.8f}, {upper:.8f}"
            f"{closing_bracket}"
        )

    interval_table = pd.DataFrame(
        {
            "interval": np.arange(
                1,
                int(n_intervals) + 1,
            ),
            "lower_edge": interval_edges[:-1],
            "upper_edge": interval_edges[1:],
            "interval_label": interval_labels,
        }
    )

    print(f"\n{description}: interval definitions")
    display(
        interval_table.style.format(
            {
                "lower_edge": "{:.8f}",
                "upper_edge": "{:.8f}",
            }
        )
    )

    n_molecules = int(
        similarity_matrix.shape[0]
    )

    total_blocks = count_upper_triangle_blocks(
        n_molecules=n_molecules,
        block_size=block_size,
    )

    priority_rng = np.random.default_rng(
        int(seed)
    )

    reservoir_pairs = [
        np.empty((0, 2), dtype=np.int64)
        for _ in range(int(n_intervals))
    ]

    reservoir_similarities = [
        np.empty(0, dtype=np.float64)
        for _ in range(int(n_intervals))
    ]

    reservoir_priorities = [
        np.empty(0, dtype=np.float64)
        for _ in range(int(n_intervals))
    ]

    candidate_counts = np.zeros(
        int(n_intervals),
        dtype=np.int64,
    )

    block_iterator = iter_upper_triangle_blocks(
        similarity_matrix=similarity_matrix,
        block_size=block_size,
    )

    for block in tqdm(
        block_iterator,
        total=total_blocks,
        desc=f"{description}: sampling intervals",
        unit="block",
        leave=True,
    ):
        values = block["values"]

        finite_mask = np.isfinite(values)
        finite_positions = np.flatnonzero(
            finite_mask
        )

        # Generate one priority for every block entry. This preserves a
        # simple reproducible RNG stream irrespective of finite-value layout.
        priorities = priority_rng.random(
            values.size
        )

        if len(finite_positions) == 0:
            continue

        finite_values = values[
            finite_positions
        ]

        finite_interval_ids = (
            assign_equal_width_intervals(
                values=finite_values,
                interval_edges=interval_edges,
            )
        )

        for interval_id in range(int(n_intervals)):
            positions_within_finite = np.flatnonzero(
                finite_interval_ids == interval_id
            )

            if len(positions_within_finite) == 0:
                continue

            positions = finite_positions[
                positions_within_finite
            ]

            candidate_counts[interval_id] += (
                len(positions)
            )

            # A block can contribute no more than the final requested
            # sample size to a global interval reservoir.
            local_keep_count = min(
                int(n_pairs_per_interval),
                len(positions),
            )

            if local_keep_count < len(positions):
                local_keep_indices = np.argpartition(
                    priorities[positions],
                    local_keep_count - 1,
                )[:local_keep_count]

                selected_positions = positions[
                    local_keep_indices
                ]

            else:
                selected_positions = positions

            candidate_pairs = block_positions_to_pairs(
                positions=selected_positions,
                row_start=block["row_start"],
                col_start=block["col_start"],
                n_block_cols=block[
                    "n_block_cols"
                ],
                local_rows=block["local_rows"],
                local_cols=block["local_cols"],
            )

            candidate_similarities = np.asarray(
                values[selected_positions],
                dtype=np.float64,
            )

            candidate_priorities = np.asarray(
                priorities[selected_positions],
                dtype=np.float64,
            )

            (
                reservoir_pairs[interval_id],
                reservoir_similarities[
                    interval_id
                ],
                reservoir_priorities[
                    interval_id
                ],
            ) = keep_smallest_priority_candidates(
                existing_pairs=reservoir_pairs[
                    interval_id
                ],
                existing_similarities=(
                    reservoir_similarities[
                        interval_id
                    ]
                ),
                existing_priorities=(
                    reservoir_priorities[
                        interval_id
                    ]
                ),
                candidate_pairs=candidate_pairs,
                candidate_similarities=(
                    candidate_similarities
                ),
                candidate_priorities=(
                    candidate_priorities
                ),
                sample_size=n_pairs_per_interval,
            )

    insufficient_intervals = np.flatnonzero(
        candidate_counts
        < int(n_pairs_per_interval)
    )

    if len(insufficient_intervals):
        details = "\n".join(
            (
                f"  Interval {int(index) + 1} "
                f"{interval_labels[index]}: "
                f"{int(candidate_counts[index]):,} available; "
                f"sampling {min(int(candidate_counts[index]), int(n_pairs_per_interval)):,}"
            )
            for index in insufficient_intervals
        )

        tqdm.write(
            f"{description}: some equal-width intervals contain "
            f"fewer than the requested {n_pairs_per_interval} pairs. "
            "All available pairs from those intervals will be used.\n"
            f"{details}"
        )

    sampled_frames = []

    for interval_id in range(int(n_intervals)):
        order = np.argsort(
            reservoir_priorities[interval_id]
        )

        pairs = reservoir_pairs[
            interval_id
        ][order]

        similarities = reservoir_similarities[
            interval_id
        ][order]

        priorities = reservoir_priorities[
            interval_id
        ][order]

        sampled_frames.append(
            pd.DataFrame(
                {
                    "interval": interval_id + 1,
                    "interval_label": (
                        interval_labels[interval_id]
                    ),
                    "mol_i": pairs[:, 0],
                    "mol_j": pairs[:, 1],
                    "similarity": similarities,
                    "sampling_priority": priorities,
                }
            )
        )

    sampled_pairs_df = pd.concat(
        sampled_frames,
        ignore_index=True,
    )

    sampled_pairs_df.insert(
        0,
        "pair_index",
        np.arange(
            len(sampled_pairs_df),
            dtype=np.int64,
        ),
    )

    finite_total = int(
        candidate_counts.sum()
    )

    sampled_counts = np.asarray(
        [
            len(reservoir_pairs[index])
            for index in range(int(n_intervals))
        ],
        dtype=np.int64,
    )

    requested_counts = np.full(
        int(n_intervals),
        int(n_pairs_per_interval),
        dtype=np.int64,
    )

    shortfalls = np.maximum(
        requested_counts - sampled_counts,
        0,
    )

    def safe_interval_stat(values, statistic):
        values = np.asarray(values)

        if values.size == 0:
            return np.nan

        return float(statistic(values))

    interval_summary = pd.DataFrame(
        {
            "interval": np.arange(
                1,
                int(n_intervals) + 1,
            ),
            "interval_label": interval_labels,
            "lower_edge": interval_edges[:-1],
            "upper_edge": interval_edges[1:],
            "candidate_pairs": candidate_counts,
            "candidate_fraction": (
                candidate_counts / finite_total
            ),
            "requested_pairs": requested_counts,
            "sampled_pairs": sampled_counts,
            "pair_shortfall": shortfalls,
            "interval_status": np.where(
                candidate_counts == 0,
                "empty",
                np.where(
                    candidate_counts
                    < int(n_pairs_per_interval),
                    "partial",
                    "complete",
                ),
            ),
            "sample_similarity_min": [
                safe_interval_stat(
                    reservoir_similarities[index],
                    np.min,
                )
                for index in range(int(n_intervals))
            ],
            "sample_similarity_median": [
                safe_interval_stat(
                    reservoir_similarities[index],
                    np.median,
                )
                for index in range(int(n_intervals))
            ],
            "sample_similarity_max": [
                safe_interval_stat(
                    reservoir_similarities[index],
                    np.max,
                )
                for index in range(int(n_intervals))
            ],
        }
    )

    return (
        sampled_pairs_df,
        interval_summary,
        interval_table,
        scan_info,
    )


## Spearman calculation

- Every .npy file enters its own sampling, conformer-generation, shape-scoring, and color-scoring loop. Separate finite-value masks and Spearman statistics are retained for shape and color.


In [ ]:
def clean_model_name(file_path):
    """Create concise labels from the expected matrix filenames."""
    stem = Path(file_path).stem

    stem = re.sub(
        (
            r"(?i)[\s_-]*embeddings"
            r"[\s_-]*cosine"
            r"[\s_-]*similarity$"
        ),
        "",
        stem,
    )

    stem = re.sub(
        r"(?i)[\s_-]*ZINC[\s_-]*30K$",
        "",
        stem,
    )

    stem = re.sub(
        r"[\s_-]+",
        " ",
        stem,
    ).strip()

    return stem


def safe_filename(text):
    cleaned = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(text),
    )

    return cleaned.strip("_")


def stable_model_seed(
    matrix_path,
    base_seed=42,
):
    filename_hash = zlib.crc32(
        Path(matrix_path).name.encode("utf-8")
    )

    return int(
        (int(base_seed) + filename_hash)
        % (2**32)
    )


def calculate_spearman(
    similarities,
    reference_scores,
):
    """ Calculate Spearman rho after excluding nonfinite observations."""
    similarities = np.asarray(
        similarities,
        dtype=np.float64,
    )

    reference_scores = np.asarray(
        reference_scores,
        dtype=np.float64,
    )

    valid_mask = (
        np.isfinite(similarities)
        & np.isfinite(reference_scores)
    )

    valid_similarities = similarities[
        valid_mask
    ]

    valid_reference_scores = reference_scores[
        valid_mask
    ]

    if len(valid_similarities) < 3:
        return np.nan, np.nan, valid_mask

    if (
        np.unique(valid_similarities).size < 2
        or np.unique(valid_reference_scores).size < 2
    ):
        return np.nan, np.nan, valid_mask

    result = spearmanr(
        valid_similarities,
        valid_reference_scores,
        nan_policy="omit",
    )

    rho = float(
        result.statistic
        if hasattr(result, "statistic")
        else result[0]
    )

    p_value = float(
        result.pvalue
        if hasattr(result, "pvalue")
        else result[1]
    )

    return rho, p_value, valid_mask


def process_similarity_folder(
    folder,
    model_type,
    smiles,
    output_dir,
    n_pairs_per_interval=100,
    n_intervals=10,
    matrix_block_size=1024,
    n_confs=5,
    n_jobs=16,
    prune_rms=0.5,
    pair_score_chunk_size=25,
    alignment_score_dtype=np.float32,
    color_opt_param=0.0,
    base_seed=42,
):
    """ Independently process every .npy matrix in one folder."""
    folder = Path(folder)
    output_dir = Path(output_dir)

    matrix_paths = sorted(
        path
        for path in folder.glob("*.npy")
        if path.is_file()
    )

    if not matrix_paths:
        raise FileNotFoundError(
            f"No .npy files were found in "
            f"{folder.resolve()}."
        )

    smiles_array = np.asarray(
        smiles,
        dtype=object,
    )

    expected_shape = (
        len(smiles_array),
        len(smiles_array),
    )

    result_rows = []
    pair_detail_frames = []
    interval_summary_frames = []
    interval_definition_frames = []

    file_progress = tqdm(
        matrix_paths,
        desc=f"{model_type} models",
        unit="model",
        leave=True,
    )

    for model_number, matrix_path in enumerate(
        file_progress,
        start=1,
    ):
        model_name = clean_model_name(
            matrix_path
        )

        file_progress.set_postfix_str(
            model_name
        )

        print("\n" + "=" * 88)
        print(
            f"{model_type} model "
            f"{model_number}/{len(matrix_paths)}: "
            f"{model_name}"
        )
        print(f"File: {matrix_path}")
        print("=" * 88)

        similarity_matrix = np.load(
            matrix_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        model_seed = stable_model_seed(
            matrix_path,
            base_seed=base_seed,
        )

        (
            sampled_pairs,
            interval_summary,
            interval_table,
            scan_info,
        ) = sample_pairs_from_equal_width_intervals(
            similarity_matrix=similarity_matrix,
            n_pairs_per_interval=(
                n_pairs_per_interval
            ),
            n_intervals=n_intervals,
            block_size=matrix_block_size,
            seed=model_seed,
            description=model_name,
        )

        print(
            f"\n{model_name}: full upper-triangle "
            "equal-width interval counts"
        )

        display(
            interval_summary[
                [
                    "interval",
                    "interval_label",
                    "candidate_pairs",
                    "candidate_fraction",
                    "requested_pairs",
                    "sampled_pairs",
                    "pair_shortfall",
                    "interval_status",
                    "sample_similarity_min",
                    "sample_similarity_median",
                    "sample_similarity_max",
                ]
            ].style.format(
                {
                    "candidate_pairs": "{:,}",
                    "candidate_fraction": "{:.3%}",
                    "sample_similarity_min": "{:.8f}",
                    "sample_similarity_median": "{:.8f}",
                    "sample_similarity_max": "{:.8f}",
                }
            )
        )

        pair_indices = sampled_pairs[
            ["mol_i", "mol_j"]
        ].to_numpy(dtype=np.int64)

        (
            returned_pairs,
            shape_scores,
            color_scores,
        ) = max_rdshapealign_scores_for_pairs_parallel(
            smiles=smiles,
            pairs=pair_indices,
            n_confs=n_confs,
            n_jobs=n_jobs,
            prune_rms=prune_rms,
            chunk_size=pair_score_chunk_size,
            dtype=alignment_score_dtype,
            color_opt_param=color_opt_param,
            progress_prefix=model_name,
        )

        if not np.array_equal(
            returned_pairs,
            pair_indices,
        ):
            raise RuntimeError(
                f"Returned pair ordering changed for "
                f"{model_name}."
            )

        sampled_pairs["shape_score"] = (
            shape_scores
        )

        sampled_pairs["color_tanimoto_score"] = (
            color_scores
        )

        sampled_pairs["smiles_i"] = (
            smiles_array[
                sampled_pairs[
                    "mol_i"
                ].to_numpy()
            ]
        )

        sampled_pairs["smiles_j"] = (
            smiles_array[
                sampled_pairs[
                    "mol_j"
                ].to_numpy()
            ]
        )

        (
            shape_rho,
            shape_p_value,
            shape_valid_mask,
        ) = calculate_spearman(
            sampled_pairs["similarity"],
            sampled_pairs["shape_score"],
        )

        (
            color_rho,
            color_p_value,
            color_valid_mask,
        ) = calculate_spearman(
            sampled_pairs["similarity"],
            sampled_pairs["color_tanimoto_score"],
        )

        sampled_pairs[
            "valid_for_shape_correlation"
        ] = shape_valid_mask

        sampled_pairs[
            "valid_for_color_correlation"
        ] = color_valid_mask

        sampled_pairs.insert(
            0,
            "model_type",
            model_type,
        )

        sampled_pairs.insert(
            1,
            "model",
            model_name,
        )

        sampled_pairs.insert(
            2,
            "matrix_file",
            matrix_path.name,
        )

        interval_width = (
            scan_info["maximum_similarity"]
            - scan_info["minimum_similarity"]
        ) / int(n_intervals)

        result_row = {
            "model_type": model_type,
            "model": model_name,
            "matrix_file": matrix_path.name,

            # Explicit shape statistics.
            "shape_spearman_rho": shape_rho,
            "shape_p_value": shape_p_value,

            # Explicit color statistics.
            "color_spearman_rho": color_rho,
            "color_p_value": color_p_value,

            "n_requested": int(
                int(n_intervals)
                * int(n_pairs_per_interval)
            ),
            "n_sampled": int(
                len(sampled_pairs)
            ),
            "n_pair_shortfall": int(
                int(n_intervals)
                * int(n_pairs_per_interval)
                - len(sampled_pairs)
            ),
            "n_intervals_below_target": int(
                (
                    interval_summary["sampled_pairs"]
                    < int(n_pairs_per_interval)
                ).sum()
            ),
            "n_empty_intervals": int(
                (
                    interval_summary["candidate_pairs"]
                    == 0
                ).sum()
            ),
            "n_valid_shape": int(
                shape_valid_mask.sum()
            ),
            "n_valid_color": int(
                color_valid_mask.sum()
            ),
            "n_missing_shape_scores": int(
                len(sampled_pairs)
                - shape_valid_mask.sum()
            ),
            "n_missing_color_scores": int(
                len(sampled_pairs)
                - color_valid_mask.sum()
            ),
            "n_intervals": int(
                n_intervals
            ),
            "n_pairs_per_interval": int(
                n_pairs_per_interval
            ),
            "minimum_similarity": float(
                scan_info["minimum_similarity"]
            ),
            "maximum_similarity": float(
                scan_info["maximum_similarity"]
            ),
            "interval_width": float(
                interval_width
            ),
            "finite_upper_triangle_pairs": int(
                scan_info[
                    "finite_upper_triangle_pairs"
                ]
            ),
            "nonfinite_upper_triangle_pairs": int(
                scan_info[
                    "nonfinite_upper_triangle_pairs"
                ]
            ),
            "color_opt_param": float(
                color_opt_param
            ),
            "sampling_seed": int(
                model_seed
            ),
        }

        result_rows.append(result_row)

        for frame in (
            interval_summary,
            interval_table,
        ):
            frame.insert(
                0,
                "model_type",
                model_type,
            )

            frame.insert(
                1,
                "model",
                model_name,
            )

            frame.insert(
                2,
                "matrix_file",
                matrix_path.name,
            )

        pair_detail_frames.append(
            sampled_pairs
        )

        interval_summary_frames.append(
            interval_summary
        )

        interval_definition_frames.append(
            interval_table
        )

        print(
            f"\n{model_name}:"
            f"\n  Shape Spearman rho = {shape_rho:.4f}; "
            f"p = {shape_p_value:.4g}; "
            f"valid pairs = "
            f"{int(shape_valid_mask.sum()):,}/"
            f"{len(sampled_pairs):,}"
            f"\n  Color Spearman rho = {color_rho:.4f}; "
            f"p = {color_p_value:.4g}; "
            f"valid pairs = "
            f"{int(color_valid_mask.sum()):,}/"
            f"{len(sampled_pairs):,}"
        )

        model_prefix = (
            f"{safe_filename(model_type)}__"
            f"{safe_filename(model_name)}"
        )

        sampled_pairs.to_csv(
            PAIR_OUTPUT_DIR
            / f"{model_prefix}__sampled_pairs_shape_color.csv",
            index=False,
        )

        interval_summary.to_csv(
            INTERVAL_OUTPUT_DIR
            / f"{model_prefix}__interval_summary.csv",
            index=False,
        )

        interval_table.to_csv(
            INTERVAL_OUTPUT_DIR
            / f"{model_prefix}__interval_definitions.csv",
            index=False,
        )

        # Save a checkpoint after every completed model.
        pd.DataFrame(result_rows).to_csv(
            output_dir
            / (
                f"{safe_filename(model_type)}"
                "__shape_color_correlation_checkpoint.csv"
            ),
            index=False,
        )

        del similarity_matrix
        gc.collect()

    results = pd.DataFrame(
        result_rows
    )

    pair_details = pd.concat(
        pair_detail_frames,
        ignore_index=True,
    )

    interval_summaries = pd.concat(
        interval_summary_frames,
        ignore_index=True,
    )

    interval_definitions = pd.concat(
        interval_definition_frames,
        ignore_index=True,
    )

    return (
        results,
        pair_details,
        interval_summaries,
        interval_definitions,
    )


## Process fingerprint matrices

In [ ]:
(
    fingerprint_results,
    fingerprint_pair_details,
    fingerprint_interval_summaries,
    fingerprint_interval_definitions,
) = process_similarity_folder(
    folder=FINGERPRINT_DIR,
    model_type="Fingerprint",
    smiles=smiles,
    output_dir=OUTPUT_DIR,
    n_pairs_per_interval=N_PAIRS_PER_INTERVAL,
    n_intervals=N_INTERVALS,
    matrix_block_size=MATRIX_BLOCK_SIZE,
    n_confs=N_CONFS,
    n_jobs=N_JOBS,
    prune_rms=PRUNE_RMS,
    pair_score_chunk_size=(
        PAIR_SCORE_CHUNK_SIZE
    ),
    alignment_score_dtype=(
        ALIGNMENT_SCORE_DTYPE
    ),
    color_opt_param=COLOR_OPT_PARAM,
    base_seed=BASE_SEED,
)

display(
    fingerprint_results.sort_values(
        "shape_spearman_rho",
        ascending=False,
    )
)


## Process embedding matrices

In [ ]:
(
    embedding_results,
    embedding_pair_details,
    embedding_interval_summaries,
    embedding_interval_definitions,
) = process_similarity_folder(
    folder=EMBEDDING_DIR,
    model_type="Embedding",
    smiles=smiles,
    output_dir=OUTPUT_DIR,
    n_pairs_per_interval=N_PAIRS_PER_INTERVAL,
    n_intervals=N_INTERVALS,
    matrix_block_size=MATRIX_BLOCK_SIZE,
    n_confs=N_CONFS,
    n_jobs=N_JOBS,
    prune_rms=PRUNE_RMS,
    pair_score_chunk_size=(
        PAIR_SCORE_CHUNK_SIZE
    ),
    alignment_score_dtype=(
        ALIGNMENT_SCORE_DTYPE
    ),
    color_opt_param=COLOR_OPT_PARAM,
    base_seed=BASE_SEED,
)

display(
    embedding_results.sort_values(
        "shape_spearman_rho",
        ascending=False,
    )
)


## Save consolidated shape and color output tables

In [14]:
all_results = pd.concat(
    [
        fingerprint_results,
        embedding_results,
    ],
    ignore_index=True,
)

all_pair_details = pd.concat(
    [
        fingerprint_pair_details,
        embedding_pair_details,
    ],
    ignore_index=True,
)

all_interval_summaries = pd.concat(
    [
        fingerprint_interval_summaries,
        embedding_interval_summaries,
    ],
    ignore_index=True,
)

all_interval_definitions = pd.concat(
    [
        fingerprint_interval_definitions,
        embedding_interval_definitions,
    ],
    ignore_index=True,
)

all_results.to_csv(
    OUTPUT_DIR
    / "all_model_shape_color_spearman_correlations.csv",
    index=False,
)

all_pair_details.to_csv(
    OUTPUT_DIR
    / "all_sampled_pairs_shape_color_scores.csv",
    index=False,
)

all_interval_summaries.to_csv(
    OUTPUT_DIR
    / "all_equal_width_interval_diagnostics.csv",
    index=False,
)

all_interval_definitions.to_csv(
    OUTPUT_DIR
    / "all_equal_width_interval_definitions.csv",
    index=False,
)

display(
    all_results.sort_values(
        "shape_spearman_rho",
        ascending=False,
    )
)


,model_type,model,matrix_file,shape_spearman_rho,shape_p_value,color_spearman_rho,color_p_value,spearman_rho,p_value,n_requested,...,n_missing_color_scores,n_intervals,n_pairs_per_interval,minimum_similarity,maximum_similarity,interval_width,finite_upper_triangle_pairs,nonfinite_upper_triangle_pairs,color_opt_param,sampling_seed
3,Embedding,COATI,COATI ZINC30K embeddings_cosine_similarity.npy,0.795216,1.210553e-208,0.771665,6.741002e-189,0.795216,1.210553e-208,1000,...,48,10,100,0.919000,0.999507,0.008051,449985000,0,0.0,194694156
0,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,0.794750,3.717490e-197,0.781629,1.745232e-186,0.794750,3.717490e-197,1000,...,99,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,829204155
1,Fingerprint,ECFP4,ECFP4 ZINC 30K.npy,0.734675,1.001525e-159,0.839213,1.350288e-249,0.734675,1.001525e-159,1000,...,57,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,2516695913
2,Fingerprint,Topological torsion,Topological torsion ZINC 30K.npy,0.734062,3.517707e-161,0.825454,3.534112e-237,0.734062,3.517707e-161,1000,...,52,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,1178484935
10,Embedding,MolFormer,MolFormer ZINC30K embeddings_cosine_similarity...,0.672553,6.551412e-106,0.711978,4.895771e-124,0.672553,6.551412e-106,1000,...,204,10,100,0.166170,0.999225,0.083305,449985000,0,0.0,803926723
4,Embedding,CheMeleon,CheMeleon ZINC30K embeddings_cosine_similarity...,0.534373,6.845077e-68,0.423880,1.006100e-40,0.534373,6.845077e-68,1000,...,96,10,100,0.093038,0.999996,0.090696,449985000,0,0.0,1673940603
6,Embedding,MiniMol,MiniMol ZINC30K embeddings_cosine_similarity.npy,0.433102,2.031634e-31,0.491199,3.370200e-41,0.433102,2.031634e-31,1000,...,343,10,100,0.301605,1.000000,0.069839,449985000,0,0.0,718175769
11,Embedding,UniMol,UniMol ZINC30K embeddings_cosine_similarity.npy,0.393387,6.413048e-19,0.313361,3.258412e-12,0.393387,6.413048e-19,1000,...,478,10,100,0.381000,1.000000,0.061900,449985000,0,0.0,1279581749
7,Embedding,MolBERT,MolBERT ZINC30K embeddings_cosine_similarity.npy,0.364485,6.111786e-31,0.447008,2.045691e-47,0.364485,6.111786e-31,1000,...,59,10,100,-0.233979,1.000000,0.123398,449985000,0,0.0,1862552228
9,Embedding,MolE,MolE ZINC30K embeddings_cosine_similarity.npy,0.341231,7.605917e-27,0.377333,6.608426e-33,0.341231,7.605917e-27,1000,...,68,10,100,-0.265658,1.000000,0.126566,449985000,0,0.0,1504671662


## Interval-occupancy check

Equal-width intervals are not expected to contain equal fractions of the pair distribution. This table shows the observed occupancy of every interval for every model.

In [15]:
interval_occupancy = (
    all_interval_summaries[
        [
            "model_type",
            "model",
            "interval",
            "interval_label",
            "candidate_fraction",
            "candidate_pairs",
            "requested_pairs",
            "sampled_pairs",
            "pair_shortfall",
            "interval_status",
        ]
    ]
    .copy()
)

display(
    interval_occupancy.style.format(
        {
            "candidate_fraction": "{:.3%}",
            "candidate_pairs": "{:,}",
            "sampled_pairs": "{:,}",
        }
    )
)


,model_type,model,interval,interval_label,candidate_fraction,candidate_pairs,requested_pairs,sampled_pairs,pair_shortfall,interval_status
0,Fingerprint,Atom pair,1,"[0.00000000, 0.10000000)",0.014%,"61,626",100,100,0,complete
1,Fingerprint,Atom pair,2,"[0.10000000, 0.20000000)",0.753%,"3,388,837",100,100,0,complete
2,Fingerprint,Atom pair,3,"[0.20000000, 0.30000000)",12.309%,"55,388,594",100,100,0,complete
3,Fingerprint,Atom pair,4,"[0.30000000, 0.40000000)",55.273%,"248,720,301",100,100,0,complete
4,Fingerprint,Atom pair,5,"[0.40000000, 0.50000000)",30.701%,"138,148,747",100,100,0,complete
5,Fingerprint,Atom pair,6,"[0.50000000, 0.60000000)",0.942%,"4,239,313",100,100,0,complete
6,Fingerprint,Atom pair,7,"[0.60000000, 0.70000000)",0.008%,"34,279",100,100,0,complete
7,Fingerprint,Atom pair,8,"[0.70000000, 0.80000000)",0.001%,"2,729",100,100,0,complete
8,Fingerprint,Atom pair,9,"[0.80000000, 0.90000000)",0.000%,410,100,100,0,complete
9,Fingerprint,Atom pair,10,"[0.90000000, 1.00000000]",0.000%,164,100,100,0,complete


## Final global and rolling Spearman figure

Load the saved correlation and pair-level CSVs so that this section can be rerun without repeating sampling, conformer generation or alignment. Calculate rolling profiles once, then display and export a single combined figure.


In [16]:
# Change this to wherever the results folder is located
OUTPUT_DIR = Path("./shape_color_similarity_equal_width_results")

# Load global Spearman correlations
all_results = pd.read_csv(
    OUTPUT_DIR / "all_model_shape_color_spearman_correlations.csv"
)

# Load pair-level similarity + shape/color scores
all_pair_details = pd.read_csv(
    OUTPUT_DIR / "all_sampled_pairs_shape_color_scores.csv"
)

print("all_results:", all_results.shape)
print("all_pair_details:", all_pair_details.shape)

display(all_results.head())
display(all_pair_details.head())

all_results: (12, 27)
all_pair_details: (11865, 16)


,model_type,model,matrix_file,shape_spearman_rho,shape_p_value,color_spearman_rho,color_p_value,spearman_rho,p_value,n_requested,...,n_missing_color_scores,n_intervals,n_pairs_per_interval,minimum_similarity,maximum_similarity,interval_width,finite_upper_triangle_pairs,nonfinite_upper_triangle_pairs,color_opt_param,sampling_seed
0,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,0.794750,3.717490e-197,0.781629,1.745232e-186,0.794750,3.717490e-197,1000,...,99,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,829204155
1,Fingerprint,ECFP4,ECFP4 ZINC 30K.npy,0.734675,1.001525e-159,0.839213,1.350288e-249,0.734675,1.001525e-159,1000,...,57,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,2516695913
2,Fingerprint,Topological torsion,Topological torsion ZINC 30K.npy,0.734062,3.517707e-161,0.825454,3.534112e-237,0.734062,3.517707e-161,1000,...,52,10,100,0.000000,1.000000,0.100000,449985000,0,0.0,1178484935
3,Embedding,COATI,COATI ZINC30K embeddings_cosine_similarity.npy,0.795216,1.210553e-208,0.771665,6.741002e-189,0.795216,1.210553e-208,1000,...,48,10,100,0.919000,0.999507,0.008051,449985000,0,0.0,194694156
4,Embedding,CheMeleon,CheMeleon ZINC30K embeddings_cosine_similarity...,0.534373,6.845077e-68,0.423880,1.006100e-40,0.534373,6.845077e-68,1000,...,96,10,100,0.093038,0.999996,0.090696,449985000,0,0.0,1673940603


,model_type,model,matrix_file,pair_index,interval,interval_label,mol_i,mol_j,similarity,sampling_priority,shape_score,color_tanimoto_score,smiles_i,smiles_j,valid_for_shape_correlation,valid_for_color_correlation
0,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,0,1,"[0.00000000, 0.10000000)",17948,18896,0.072464,2.920855e-07,0.581702,0.126789,COC[C@H](CNC(=O)COc1ccccc1)NC(=O)c1cc(C(C)=O)cn1C,OCCC1(CNC[C@@H](O)C2CC2)CCCCC1,True,True
1,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,1,1,"[0.00000000, 0.10000000)",1836,3726,0.093333,4.020581e-06,0.703450,0.145286,CCCN1CC[C@H](N[C@H]2C[C@H]3CC[C@@H]2[C@@H](N)C...,C[C@H]1Oc2ccccc2[C@H]1NC(=O)Nc1ccc(NS(N)(=O)=O...,True,True
2,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,2,1,"[0.00000000, 0.10000000)",25914,28978,0.056604,1.011211e-05,NaN,NaN,C[C@@H]1CCc2[nH]c(=NCC[C@H]3CCCOC3)nc(N(C)Cc3n...,BrC1C[C@@H]2CC[C@@H](C1)C2,False,False
3,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,3,1,"[0.00000000, 0.10000000)",5436,18896,0.072131,2.147590e-05,NaN,NaN,C[C@H](CN(C)C(=O)c1nc(Cl)ccc1N)N(C)C(=O)C1[C@@...,OCCC1(CNC[C@@H](O)C2CC2)CCCCC1,False,False
4,Fingerprint,Atom pair,Atom pair ZINC 30K.npy,4,1,"[0.00000000, 0.10000000)",9762,23527,0.078431,4.054624e-05,0.523655,0.144879,CC(=CC1CC1)C(=O)N[C@@H](C)Cn1cc(CNC(=O)Cc2cncc...,OCCC1(CN[C@H]2CCO[C@@]23CCOC3)CC1,True,True


In [17]:
def rolling_shape_color_spearman(
    data,
    window=200,
    step=25,
    n_boot=300,
    min_unique_similarity=10,
    seed=42,
):
    """
    Calculate rolling shape and color Spearman correlations using
    identical pairs and identical windows.
    """
    columns = [
        "similarity",
        "shape_score",
        "color_tanimoto_score",
    ]

    missing = set(columns).difference(data.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    # Requiring both scores ensures that shape and color are compared
    # on exactly the same molecular pairs.
    ordered = (
        data[columns]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values("similarity")
        .reset_index(drop=True)
    )

    if len(ordered) < window:
        raise ValueError(
            f"Only {len(ordered)} complete pairs are available, "
            f"which is fewer than window={window}."
        )

    rng = np.random.default_rng(seed)
    rows = []

    for start in range(
        0,
        len(ordered) - window + 1,
        step,
    ):
        stop = start + window
        current = ordered.iloc[start:stop]

        x = current["similarity"].to_numpy()
        shape = current["shape_score"].to_numpy()
        color = current[
            "color_tanimoto_score"
        ].to_numpy()

        n_unique_similarity = np.unique(x).size

        if n_unique_similarity < min_unique_similarity:
            shape_rho = np.nan
            color_rho = np.nan

            shape_lower = np.nan
            shape_upper = np.nan
            color_lower = np.nan
            color_upper = np.nan

        else:
            shape_rho = float(
                spearmanr(x, shape).statistic
            )

            color_rho = float(
                spearmanr(x, color).statistic
            )

            shape_bootstrap = []
            color_bootstrap = []

            for _ in range(n_boot):
                indices = rng.integers(
                    0,
                    len(current),
                    len(current),
                )

                bootstrap_x = x[indices]
                bootstrap_shape = shape[indices]
                bootstrap_color = color[indices]

                if np.unique(bootstrap_x).size < 2:
                    continue

                if np.unique(bootstrap_shape).size >= 2:
                    shape_bootstrap.append(
                        spearmanr(
                            bootstrap_x,
                            bootstrap_shape,
                        ).statistic
                    )

                if np.unique(bootstrap_color).size >= 2:
                    color_bootstrap.append(
                        spearmanr(
                            bootstrap_x,
                            bootstrap_color,
                        ).statistic
                    )

            if shape_bootstrap:
                shape_lower, shape_upper = np.quantile(
                    shape_bootstrap,
                    [0.025, 0.975],
                )
            else:
                shape_lower = np.nan
                shape_upper = np.nan

            if color_bootstrap:
                color_lower, color_upper = np.quantile(
                    color_bootstrap,
                    [0.025, 0.975],
                )
            else:
                color_lower = np.nan
                color_upper = np.nan

        rows.append(
            {
                "window_start": start,
                "window_stop": stop,
                "sample_percentile": (
                    100
                    * (start + window / 2)
                    / len(ordered)
                ),
                "similarity_centre": np.median(x),
                "similarity_min": np.min(x),
                "similarity_max": np.max(x),
                "shape_rho": shape_rho,
                "shape_lower": shape_lower,
                "shape_upper": shape_upper,
                "color_rho": color_rho,
                "color_lower": color_lower,
                "color_upper": color_upper,
                "n_pairs": len(current),
                "n_unique_similarity": (
                    n_unique_similarity
                ),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
MODEL_ORDER = [
    "COATI",
    "Atom pair",
    "ECFP4",
    "Topological torsion",
    "MolFormer",
    "CheMeleon",
    "MolBERT",
    "MiniMol",
    "UniMol",
    "MolE",
    "ChemBERTa",
    "MolCLR",
]


FINGERPRINT_MODELS = {"ECFP4", "Atom pair", "Topological torsion"}


def _contiguous_group_runs(models):
    """
    Return (start_index, end_index, label) for adjacent model classes.

    Sorting by global color correlation can interleave fingerprints and
    learned embeddings. Separate braces prevent a class label from wrongly
    spanning a model of the other class.
    """
    runs = []
    if not models:
        return runs

    def group_label(model):
        return (
            "Fingerprints"
            if model in FINGERPRINT_MODELS
            else "Learned embeddings"
        )

    start = 0
    current_label = group_label(models[0])
    for index in range(1, len(models)):
        next_label = group_label(models[index])
        if next_label != current_label:
            runs.append((start, index - 1, current_label))
            start = index
            current_label = next_label
    runs.append((start, len(models) - 1, current_label))
    return runs


def _add_group_underbrace(fig, start, end, n_models, axis_bottom, label):
    """
    Draw a black underbrace below the global bar chart's x-axis labels.

    Both x and y use paper coordinates so the braces sit below the *tick
    labels*, not at an arbitrary Spearman correlation value. The bracket
    spans exactly one contiguous run of representations.
    """
    x_left, x_right = fig.layout.xaxis.domain
    x_span = x_right - x_left

    # Extend almost to the category boundaries, without touching neighbours.
    x0 = x_left + x_span * (start + 0.12) / n_models
    x1 = x_left + x_span * (end + 0.88) / n_models
    middle = (x0 + x1) / 2
    width = x1 - x0

    edge_y = axis_bottom - 0.045
    shoulder_y = edge_y - 0.009
    tip_y = edge_y - 0.023

    path = (
        f"M {x0:.6f},{edge_y:.6f} "
        f"Q {x0:.6f},{shoulder_y:.6f} "
        f"{x0 + 0.12 * width:.6f},{shoulder_y:.6f} "
        f"L {middle - 0.13 * width:.6f},{shoulder_y:.6f} "
        f"Q {middle - 0.04 * width:.6f},{shoulder_y:.6f} "
        f"{middle:.6f},{tip_y:.6f} "
        f"Q {middle + 0.04 * width:.6f},{shoulder_y:.6f} "
        f"{middle + 0.13 * width:.6f},{shoulder_y:.6f} "
        f"L {x1 - 0.12 * width:.6f},{shoulder_y:.6f} "
        f"Q {x1:.6f},{shoulder_y:.6f} "
        f"{x1:.6f},{edge_y:.6f}"
    )

    fig.add_shape(
        type="path",
        path=path,
        xref="paper",
        yref="paper",
        fillcolor="rgba(0, 0, 0, 0)",
        line=dict(color="#000000", width=1.4, dash="solid"),
        opacity=1.0,
        layer="above",
    )
    fig.add_annotation(
        x=middle,
        xref="paper",
        y=tip_y - 0.006,
        yref="paper",
        text=f"<b>{label}</b>",
        opacity=1.0,
        showarrow=False,
        xanchor="center",
        yanchor="top",
        font=dict(family="Arial", size=17, color="#000000"),
    )


def plot_global_bar_and_rolling_spearman(
    results,
    profiles,
    model_order=MODEL_ORDER,
    show_confidence=False,
    bar_to_grid_gap=0.075,
):
    """
    Plot global Spearman bars and 12 rolling-correlation panels.

    Models are sorted by descending global color Spearman correlation.
    Fingerprints and learned embeddings both use solid bars, with an
    underbrace labelled by representation class below each contiguous run.

    Parameters
    ----------
    results : pandas.DataFrame
        Requires model, shape_spearman_rho, color_spearman_rho.
    profiles : dict[str, pandas.DataFrame]
        Each profile requires sample_percentile, shape_rho, color_rho,
        similarity_centre, similarity_min, similarity_max, n_pairs, and
        n_unique_similarity. Optionally window_used and (when
        show_confidence=True) shape_upper, shape_lower, color_upper,
        color_lower.
    model_order : list[str]
        Which representations are included. Display order is determined by
        descending global color correlation.
    show_confidence : bool
        Display rolling confidence bands if True.
    bar_to_grid_gap : float
        Extra paper-coordinate gap beneath the global bar chart. A minimum
        of 0.075 is reserved for the underbraces and their labels.
    """
    required_result_columns = {
        "model", "shape_spearman_rho", "color_spearman_rho"
    }
    missing = required_result_columns.difference(results.columns)
    if missing:
        raise ValueError(f"results is missing columns: {sorted(missing)}")


    shape_bar_color = "rgba(78, 134, 188, 0.50)"
    color_bar_color = "rgba(111, 168, 182, 0.50)"


    shape_rolling_color = "rgba(45, 100, 158, 0.95)"
    color_rolling_color = "rgba(52, 122, 137, 0.95)"

    # Light confidence bands, visually secondary to the rolling lines.
    shape_ci_color = "rgba(45, 100, 158, 0.07)"
    color_ci_color = "rgba(52, 122, 137, 0.07)"
    axis_color = "rgba(25, 25, 25, 0.78)"

    result_lookup = (
        results[["model", "shape_spearman_rho", "color_spearman_rho"]]
        .drop_duplicates("model")
        .set_index("model")
    )

    models = [
        model for model in model_order
        if model in profiles and model in result_lookup.index
    ]

    models = sorted(
        models,
        key=lambda model: float(result_lookup.loc[model, "color_spearman_rho"]),
        reverse=True,
    )

    shape_values = [
        float(result_lookup.loc[model, "shape_spearman_rho"])
        for model in models
    ]
    color_values = [
        float(result_lookup.loc[model, "color_spearman_rho"])
        for model in models
    ]

    # Braces are below the x-axis.
    finite_global = np.asarray(shape_values + color_values, dtype=float)
    finite_global = finite_global[np.isfinite(finite_global)]
    highest_bar = max(0.0, float(finite_global.max())) if finite_global.size else 0.0
    bar_y_max = max(0.9, highest_bar + 0.07)

    specs = [
        [{"colspan": 4}, None, None, None],
        [{}, {}, {}, {}],
        [{}, {}, {}, {}],
        [{}, {}, {}, {}],
    ]
    fig = make_subplots(
        rows=4,
        cols=4,
        specs=specs,
        subplot_titles=[""] + models,
        row_heights=[0.42, 0.193, 0.193, 0.193],
        horizontal_spacing=0.035,
        vertical_spacing=0.065,
    )

    # ------------------------------------------------------------------
    # (a) Global correlations
    # ------------------------------------------------------------------
    for series_name, values, bar_color, series_label in (
        ("Shape", shape_values, shape_bar_color, "shape"),
        ("Color", color_values, color_bar_color, "color"),
    ):
        fig.add_trace(
            go.Bar(
                x=models,
                y=values,
                name=series_name,
                marker=dict(
                    color=bar_color,
                    line=dict(color="black", width=1.0),
                ),
                text=values,
                texttemplate="%{text:.2f}",
                textposition="outside",
                textfont=dict(size=15, color="black"),
                cliponaxis=False,
                showlegend=False,
                hovertemplate=(
                    "<b>%{x}</b>"
                    f"<br>Global {series_label} ρ: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=1,
        )

    # ------------------------------------------------------------------
    # (b-m) Rolling-correlation panels.
    # ------------------------------------------------------------------
    for position, model in enumerate(models):
        row_number = position // 4 + 2
        column_number = position % 4 + 1
        profile = profiles[model]
        x = profile["sample_percentile"]

        if show_confidence:
            for prefix, ci_color in (
                ("shape", shape_ci_color),
                ("color", color_ci_color),
            ):
                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=profile[f"{prefix}_upper"],
                        mode="lines",
                        line=dict(width=0),
                        hoverinfo="skip",
                        showlegend=False,
                    ),
                    row=row_number,
                    col=column_number,
                )
                fig.add_trace(
                    go.Scatter(
                        x=x,
                        y=profile[f"{prefix}_lower"],
                        mode="lines",
                        line=dict(width=0),
                        fill="tonexty",
                        fillcolor=ci_color,
                        hoverinfo="skip",
                        showlegend=False,
                    ),
                    row=row_number,
                    col=column_number,
                )

        if "window_used" in profile.columns:
            window_used = profile["window_used"].to_numpy()
        else:
            window_used = np.full(len(profile), np.nan)

        customdata = np.column_stack([
            profile["similarity_centre"],
            profile["similarity_min"],
            profile["similarity_max"],
            profile["n_pairs"],
            profile["n_unique_similarity"],
            window_used,
        ])

        for series_name, prefix, line_color in (
            ("Shape", "shape", shape_rolling_color),
            ("Color", "color", color_rolling_color),
        ):
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=profile[f"{prefix}_rho"],
                    mode="lines",
                    name=series_name,
                    showlegend=False,
                    line=dict(
                        width=2.6,
                        color=line_color,
                        dash="solid",
                    ),
                    customdata=customdata,
                    hovertemplate=(
                        f"<b>{model}: {prefix}</b>"
                        "<br>Sample position: %{x:.1f}%"
                        "<br>Rolling ρ: %{y:.3f}"
                        "<br>Median similarity: %{customdata[0]:.5f}"
                        "<br>Similarity window: "
                        "%{customdata[1]:.5f}–%{customdata[2]:.5f}"
                        "<br>Pairs: %{customdata[3]:.0f}"
                        "<br>Unique similarities: %{customdata[4]:.0f}"
                        "<br>Window size: %{customdata[5]:.0f}"
                        "<extra></extra>"
                    ),
                ),
                row=row_number,
                col=column_number,
            )

    # ------------------------------------------------------------------
    # Large square legend markers, matching the global bar chart colors.
    # ------------------------------------------------------------------
    for series_name, bar_color in (
        ("Shape", shape_bar_color),
        ("Color", color_bar_color),
    ):
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=series_name,
                marker=dict(
                    symbol="square",
                    size=25,
                    color=bar_color,
                    line=dict(color="black", width=1.2),
                ),
                hoverinfo="skip",
                showlegend=True,
            ),
            row=1,
            col=1,
        )

    # ------------------------------------------------------------------
    # Axes.
    # ------------------------------------------------------------------
    bar_ticktext = [
        "Topological<br>torsion" if model == "Topological torsion" else model
        for model in models
    ]
    fig.update_xaxes(
        tickmode="array",
        tickvals=models,
        ticktext=bar_ticktext,
        tickfont=dict(size=16, color=axis_color),
        tickangle=0,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=axis_color,
        linewidth=1,
        ticks="outside",
        ticklen=5,
        tickwidth=1.2,
        tickcolor=axis_color,
        automargin=True,
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text="Global Spearman ρ",
        range=[0, bar_y_max],
        tickmode="linear",
        tick0=0,
        dtick=0.1,
        tickformat=".1f",
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=axis_color,
        linewidth=1,
        ticks="outside",
        ticklen=5,
        tickwidth=1.2,
        tickcolor=axis_color,
        title_font=dict(size=17, color="rgba(20, 20, 20, 0.95)"),
        tickfont=dict(size=14, color=axis_color),
        row=1,
        col=1,
    )

    for row_number in range(2, 5):
        for column_number in range(1, 5):
            fig.update_xaxes(
                range=[0, 100],
                tickmode="linear",
                tick0=0,
                dtick=25,
                showticklabels=True,
                tickfont=dict(size=14, color=axis_color),
                showgrid=False,
                zeroline=False,
                showline=True,
                linecolor=axis_color,
                linewidth=1,
                ticks="outside",
                ticklen=5,
                tickwidth=1.2,
                tickcolor=axis_color,
                row=row_number,
                col=column_number,
            )
            fig.update_yaxes(
                range=[-0.6, 0.9],
                tickmode="linear",
                tick0=-0.6,
                dtick=0.2,
                tickformat=".1f",
                showticklabels=True,
                tickfont=dict(size=14, color=axis_color),
                showgrid=False,
                zeroline=False,
                showline=True,
                linecolor=axis_color,
                linewidth=1,
                ticks="outside",
                ticklen=5,
                tickwidth=1.2,
                tickcolor=axis_color,
                row=row_number,
                col=column_number,
            )

    for column_number in range(1, 5):
        fig.update_xaxes(
            title_text="Position in sampled similarity range (%)",
            title_font=dict(size=15, color="rgba(20, 20, 20, 0.95)"),
            title_standoff=10,
            row=4,
            col=column_number,
        )
    for row_number in range(2, 5):
        fig.update_yaxes(
            title_text="Rolling Spearman ρ",
            title_font=dict(size=15, color="rgba(20, 20, 20, 0.95)"),
            title_standoff=10,
            row=row_number,
            col=1,
        )

    fig.update_annotations(
        font=dict(size=17, color="black")
    )

    bar_domain = list(fig.layout.yaxis.domain)
    maximum_new_bottom      = bar_domain[1] - 0.08
    effective_gap           = max(bar_to_grid_gap, 0.075)
    new_bar_bottom          = min(bar_domain[0] + effective_gap, maximum_new_bottom)
    fig.layout.yaxis.domain = [new_bar_bottom, bar_domain[1]]

    for start, end, label in _contiguous_group_runs(models):
        _add_group_underbrace(
            fig, start, end, len(models), new_bar_bottom, label
        )

    # Panel labels a-m.
    panel_letters = list(string.ascii_lowercase[: len(models) + 1])
    for panel_index, panel_letter in enumerate(panel_letters):
        axis_number = panel_index + 1
        xaxis_name = "xaxis" if axis_number == 1 else f"xaxis{axis_number}"
        yaxis_name = "yaxis" if axis_number == 1 else f"yaxis{axis_number}"
        x_domain = getattr(fig.layout, xaxis_name).domain
        y_domain = getattr(fig.layout, yaxis_name).domain
        fig.add_annotation(
            x=x_domain[0] - 0.014,
            y=y_domain[1] + 0.010,
            xref="paper",
            yref="paper",
            text=f"<b>{panel_letter}</b>",
            showarrow=False,
            xanchor="right",
            yanchor="bottom",
            font=dict(size=17, color="black"),
        )

    fig.update_layout(
        template="simple_white",
        barmode="group",
        bargap=0.18,
        bargroupgap=0.04,
        uniformtext_minsize=10,
        uniformtext_mode="show",
        width=1400,
        height=1500,
        hovermode="closest",
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(
            family="Arial",
            size=14,
            color="rgba(25, 25, 25, 0.95)",
        ),
        legend=dict(
            # Vertically stacked square markers, lowered into the
            # upper-right corner of the global bar-chart panel.
            orientation="v",
            yanchor="top",
            y=0.965,
            xanchor="right",
            x=0.99,
            traceorder="normal",
            itemsizing="constant",
            font=dict(size=17),
            bgcolor="rgba(255, 255, 255, 0.9)",
            borderwidth=0,
        ),
        margin=dict(t=90, l=115, r=40, b=110),
    )
    return fig

In [ ]:
def calculate_all_shape_color_rolling_spearman(
    pair_details,
    window=200,
    step=25,
    n_boot=300,
    model_order=MODEL_ORDER,
):
    """Calculate rolling shape/color Spearman profiles for all models."""

    required_columns = {
        "model",
        "similarity",
        "shape_score",
        "color_tanimoto_score",
    }

    missing = required_columns.difference(pair_details.columns)

    if missing:
        raise ValueError(
            f"pair_details is missing columns: {sorted(missing)}"
        )

    available_models = set(
        pair_details["model"].dropna().unique()
    )

    models = [
        model
        for model in model_order
        if model in available_models
    ]

    profiles = {}

    for model in models:
        model_data = pair_details.loc[
            pair_details["model"] == model
        ].copy()

        n_complete = (
            model_data[
                [
                    "similarity",
                    "shape_score",
                    "color_tanimoto_score",
                ]
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .shape[0]
        )

        model_window = min(
            int(window),
            max(50, n_complete // 3),
        )

        if model_window >= n_complete:
            model_window = n_complete - 1

        if model_window < 20:
            print(
                f"Skipping {model}: only {n_complete} "
                "complete shape/color pairs."
            )
            continue

        model_step = min(
            int(step),
            max(1, model_window // 5),
        )

        profiles[model] = rolling_shape_color_spearman(
            model_data,
            window=model_window,
            step=model_step,
            n_boot=n_boot,
            seed=42,
        )

        profiles[model]["window_used"] = model_window

    if not profiles:
        raise ValueError(
            "No model had enough complete observations."
        )

    return profiles

In [20]:
# Compute rolling profiles without constructing an unused intermediate figure.
combined_profiles = calculate_all_shape_color_rolling_spearman(
    pair_details=all_pair_details,
    window=200,
    step=25,
    n_boot=300,
)

# One combined figure: global Spearman bars + rolling correlations.
full_figure = plot_global_bar_and_rolling_spearman(
    results=all_results,
    profiles=combined_profiles,
    show_confidence=True,
)

full_figure.show()


In [23]:
# Export the same final figure as both a vector PDF and a PNG.
export_dir = Path('/home/blake/Downloads')
export_dir.mkdir(parents=True, exist_ok=True)

pdf_path = export_dir / 'rolling_shape_color_spearman.pdf'
png_path = export_dir / 'rolling_shape_color_spearman.png'

full_figure.write_image(
    str(pdf_path),
    format='pdf',
    width=1400,
    height=1500,
)
full_figure.write_image(
    str(png_path),
    format='png',
    width=1400,
    height=1500,
)